# Número Efetivo de Candidatos em Recursos (NECr) — versão gráfica

Versão visual de `3_necr.ipynb`: as mesmas análises, mas apresentadas em gráficos
(Plotly) em vez de tabelas impressas. Mesma definição e mesmo balizador:

$$\text{NECr} = \frac{1}{\sum_i p_i^2}, \qquad p_i = \texttt{prop\_vr\_receita\_candidato}$$

**Balizador: magnitude partidária ($M_p$), não magnitude do distrito.** $M_p$ é a bancada
estadual do partido na UF no dia anterior ao início das convenções partidárias (Crisp et al.
2007), via `data/processed/bancada_partido_uf.csv` — não `qt_vaga` (magnitude do distrito,
igual para todos os partidos da UF). A regra $M_p$+1 (Cox 1997) prevê concentração de
recursos em torno de $M_p$+1 candidatos viáveis, logo `NECr/(Mp+1) ≈ 1` sob coordenação.

In [1]:
import os
from pathlib import Path
ROOT = Path().resolve().parent  # notebooks/ -> project root
os.chdir(ROOT)

In [2]:
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

sys.path.insert(0, str(Path("src/2_gold").resolve()))
from cap3_cs_features import gerar_features

rrd_raw = pd.read_parquet("data/processed/rrd_df_novo.parquet")
rrd = rrd_raw[rrd_raw.ano_eleicao.isin([2018, 2022])].copy()
rrd = gerar_features(rrd)
rrd["ano_eleicao"] = rrd["ano_eleicao"].astype(str)

len(rrd)

17305

## 1. Magnitude partidária ($M_p$): distribuição da bancada estadual prévia

In [3]:
bancada = pd.read_csv("data/processed/bancada_partido_uf.csv")
harmonizacao = {"PP**": "PP", "PCdoB": "PC do B", "PTdoB": "PT do B", "SD": "SOLIDARIEDADE"}
bancada["sg_partido"] = bancada["sg_partido"].replace(harmonizacao)

bancada_uf = (
    bancada[bancada.ano_eleicao.isin([2018, 2022])]
    [["ano_eleicao", "sg_uf", "sg_partido", "n_deputados"]]
    .rename(columns={"n_deputados": "Mp"})
)
bancada_uf["ano_eleicao"] = bancada_uf["ano_eleicao"].astype(str)

rrd = rrd.merge(bancada_uf, on=["ano_eleicao", "sg_uf", "sg_partido"], how="left")
rrd["Mp"] = rrd["Mp"].fillna(0).astype(int)

listas_mp = rrd.drop_duplicates(["ano_eleicao", "sg_uf", "sg_partido"])[
    ["ano_eleicao", "sg_uf", "sg_partido", "Mp"]
]

In [4]:
def faixa_mp(mp):
    return str(mp) if mp <= 4 else "5+"

listas_mp = listas_mp.copy()
listas_mp["Mp_faixa"] = listas_mp["Mp"].apply(faixa_mp)
ordem_faixa = ["0", "1", "2", "3", "4", "5+"]

freq = (
    listas_mp.groupby(["ano_eleicao", "Mp_faixa"], observed=True)
    .size().reset_index(name="n_listas")
)
freq["Mp_faixa"] = pd.Categorical(freq["Mp_faixa"], categories=ordem_faixa, ordered=True)
freq = freq.sort_values(["ano_eleicao", "Mp_faixa"])
freq["pct"] = freq["n_listas"] / freq.groupby("ano_eleicao")["n_listas"].transform("sum") * 100

fig = px.bar(
    freq, x="Mp_faixa", y="pct", color="ano_eleicao", barmode="group",
    text=freq["pct"].round(1).astype(str) + "%",
    category_orders={"Mp_faixa": ordem_faixa},
    labels={"Mp_faixa": "Mp (bancada estadual prévia)", "pct": "% das listas", "ano_eleicao": "Ano"},
    title="Distribuição de Mp por lista (partido × UF) — 2018 vs. 2022",
)
fig.update_traces(textposition="outside")
fig.update_layout(template="plotly_white", width=900, height=500)
fig.show()

## 2. Cálculo do NECr por lista

In [5]:
def agregar_lista(g):
    total_rec = g["vr_receita_recursos_partidos"].sum()
    n_fundados = (g["vr_receita_recursos_partidos"] > 0).sum()
    sum_sq = (g["prop_vr_receita_candidato"] ** 2).sum()
    return pd.Series({
        "Mp":           g["Mp"].iloc[0],
        "qt_vaga":      g["qt_vaga"].iloc[0],
        "dm_cat":       g["dm_cat"].iloc[0],
        "n_cands":      len(g),
        "n_fundados":   int(n_fundados),
        "total_rec":    total_rec,
        "sum_sq_prop":  sum_sq,
        "NECr":         (1 / sum_sq) if sum_sq > 0 else np.nan,
    })

listas = (
    rrd
    .groupby(["ano_eleicao", "sg_uf", "sg_partido"], observed=True)
    .apply(agregar_lista, include_groups=False)
    .reset_index()
)

listas_rec = listas[listas["total_rec"] > 0].copy()
listas_rec["NECr_sobre_Mp"]   = np.where(listas_rec["Mp"] > 0, listas_rec["NECr"] / listas_rec["Mp"], np.nan)
listas_rec["NECr_sobre_Mpp1"] = listas_rec["NECr"] / (listas_rec["Mp"] + 1)
listas_rec["NECr_sobre_N"]    = listas_rec["NECr"] / listas_rec["n_cands"]
listas_rec["Mp_grupo"] = np.where(listas_rec["Mp"] == 0, "Mp = 0", "Mp > 0")

len(listas_rec)

1434

## 3. Estatísticas descritivas gerais

In [6]:
metricas = {
    "NECr": "NECr",
    "NECr_sobre_Mp": "NECr / Mp (apenas Mp>0)",
    "NECr_sobre_Mpp1": "NECr / (Mp+1)",
    "NECr_sobre_N": "NECr / N",
}
long_geral = listas_rec.melt(
    id_vars=["ano_eleicao"], value_vars=list(metricas), var_name="metrica", value_name="valor"
).dropna()
long_geral["metrica"] = long_geral["metrica"].map(metricas)

fig = px.box(
    long_geral, x="metrica", y="valor", points=False,
    title="Distribuição das métricas de NECr — todas as listas com recursos (2018+2022)",
    labels={"metrica": "", "valor": "Valor"},
)
fig.update_layout(template="plotly_white", width=900, height=500)
fig.show()

## 4. Comparação 2018 × 2022

In [7]:
fig = px.box(
    long_geral, x="metrica", y="valor", color="ano_eleicao", points=False,
    title="Métricas de NECr por ano",
    labels={"metrica": "", "valor": "Valor", "ano_eleicao": "Ano"},
)
fig.update_layout(template="plotly_white", width=950, height=550, boxmode="group")
fig.show()

## 5. NECr por magnitude do distrito (estratificação descritiva secundária)

`dm_cat` (via `qt_vaga`) é mantida só como controle descritivo do tamanho do distrito — não
é mais o balizador de nenhuma razão NECr/M.

In [8]:
from plotly.subplots import make_subplots

ordem_dm = ["Pequeno (8–12)", "Médio (16–31)", "Grande (39–70)"]

resumo_dm = (
    listas_rec.groupby(["ano_eleicao", "dm_cat"], observed=True)
    .agg(NECr_media=("NECr", "mean"), Mpp1_media=("NECr_sobre_Mpp1", "mean"), Mp_mediana=("Mp", "median"))
    .reset_index()
)
resumo_dm["dm_cat"] = pd.Categorical(resumo_dm["dm_cat"], categories=ordem_dm, ordered=True)
resumo_dm = resumo_dm.sort_values(["ano_eleicao", "dm_cat"])

fig = make_subplots(rows=1, cols=2, subplot_titles=["NECr (média)", "NECr / (Mp+1) (média)"])
cores = {"2018": "#1f4e79", "2022": "#c55a11"}
for ano in ["2018", "2022"]:
    sub = resumo_dm[resumo_dm.ano_eleicao == ano]
    fig.add_trace(go.Bar(name=ano, x=sub["dm_cat"], y=sub["NECr_media"], marker_color=cores[ano],
                          legendgroup=ano, showlegend=True), row=1, col=1)
    fig.add_trace(go.Bar(name=ano, x=sub["dm_cat"], y=sub["Mpp1_media"], marker_color=cores[ano],
                          legendgroup=ano, showlegend=False), row=1, col=2)

fig.add_hline(y=1, line_dash="dot", line_color="gray", row=1, col=2)
fig.update_layout(
    barmode="group", template="plotly_white", width=1100, height=500,
    title="NECr e NECr/(Mp+1) por magnitude do distrito × ano",
    legend_title="Ano",
)
fig.show()

## 6. Distribuição de NECr/($M_p$+1) — geral vs. $M_p$=0 vs. $M_p$>0

Se coordenação = $M_p$+1, esperamos `NECr/(Mp+1)` ≈ 1 (linha tracejada). A hipótese só é
diretamente testável onde $M_p$>0 (partido tem bancada a defender); para $M_p$=0 a previsão
$M_p$+1=1 significa "concentrar em um único candidato puxador".

In [9]:
fig = px.violin(
    listas_rec, x="ano_eleicao", y="NECr_sobre_Mpp1", color="Mp_grupo",
    box=True, points=False, log_y=True,
    category_orders={"ano_eleicao": ["2018", "2022"], "Mp_grupo": ["Mp = 0", "Mp > 0"]},
    labels={"NECr_sobre_Mpp1": "NECr / (Mp+1)  [escala log]", "ano_eleicao": "Ano", "Mp_grupo": ""},
    title="NECr / (Mp+1): geral, Mp=0 e Mp>0, por ano",
)
fig.add_hline(y=1, line_dash="dot", line_color="black",
              annotation_text="previsão teórica (=1)", annotation_position="top left")
fig.update_layout(template="plotly_white", width=950, height=550, violinmode="group")
fig.show()

## 7. Mapa-resumo: NECr/($M_p$+1) mediana por magnitude do distrito × ano

In [10]:
tab_resumo = (
    listas_rec.groupby(["ano_eleicao", "dm_cat"], observed=True)["NECr_sobre_Mpp1"]
    .median().reset_index()
)
tab_resumo["dm_cat"] = pd.Categorical(tab_resumo["dm_cat"], categories=ordem_dm, ordered=True)
pivot = tab_resumo.pivot(index="dm_cat", columns="ano_eleicao", values="NECr_sobre_Mpp1").loc[ordem_dm]

fig = px.imshow(
    pivot, text_auto=".2f", color_continuous_scale="RdYlBu_r", aspect="auto",
    labels=dict(x="Ano", y="Magnitude do distrito", color="NECr/(Mp+1)\n(mediana)"),
    title="NECr/(Mp+1) mediana por magnitude do distrito × ano",
)
fig.update_layout(template="plotly_white", width=650, height=450)
fig.show()

## 8. Proporção de listas abaixo de $M_p$+1 e 2×($M_p$+1) — por magnitude do distrito × ano

In [11]:
resultados = []
for ano in ["2018", "2022"]:
    for dm in ordem_dm + ["Total"]:
        sub = listas_rec[listas_rec.ano_eleicao == ano]
        if dm != "Total":
            sub = sub[sub.dm_cat == dm]
        if len(sub) == 0:
            continue
        mpp1 = sub["Mp"] + 1
        resultados.append({
            "Ano": ano, "Magnitude": dm, "n": len(sub),
            "< Mp+1":    (sub["NECr"] < mpp1).mean() * 100,
            "< 2(Mp+1)": (sub["NECr"] < 2 * mpp1).mean() * 100,
        })
prop_tab = pd.DataFrame(resultados)
prop_tab["Magnitude"] = pd.Categorical(prop_tab["Magnitude"], categories=ordem_dm + ["Total"], ordered=True)

long_prop = prop_tab.melt(id_vars=["Ano", "Magnitude", "n"], value_vars=["< Mp+1", "< 2(Mp+1)"],
                           var_name="limiar", value_name="pct")

fig = px.bar(
    long_prop, x="Magnitude", y="pct", color="limiar", barmode="group", facet_col="Ano",
    text=long_prop["pct"].round(1).astype(str) + "%",
    labels={"pct": "% das listas", "limiar": "Limiar"},
    title="Proporção de listas com NECr abaixo dos limiares teóricos",
)
fig.update_traces(textposition="outside")
fig.update_layout(template="plotly_white", width=1100, height=500)
fig.show()

## 9. NECr vs. $M_p$: dispersão geral com referência teórica $M_p$+1

Cada ponto é uma lista (partido × UF); passe o mouse para identificar partido/UF. A linha
tracejada é a previsão teórica NECr = $M_p$+1.

In [12]:
fig = px.scatter(
    listas_rec, x="Mp", y="NECr", color="dm_cat", facet_col="ano_eleicao",
    hover_data=["sg_uf", "sg_partido", "n_cands", "n_fundados"],
    category_orders={"dm_cat": ordem_dm, "ano_eleicao": ["2018", "2022"]},
    labels={"Mp": "Mp (bancada estadual prévia)", "NECr": "NECr", "dm_cat": "Magnitude do distrito"},
    title="NECr vs. Mp, com referência teórica NECr = Mp+1",
    opacity=0.65,
)
mp_max = listas_rec["Mp"].max()
for ax in ["x", "x2"]:
    fig.add_trace(go.Scatter(
        x=[0, mp_max], y=[1, mp_max + 1], mode="lines", line=dict(dash="dash", color="black"),
        name="NECr = Mp+1", showlegend=(ax == "x"),
    ), row=1, col=1 if ax == "x" else 2)
fig.update_layout(template="plotly_white", width=1150, height=550)
fig.show()

## 10. NECr × N fundados × $M_p$ — matriz de correlação por ano

In [13]:
cols_corr = ["NECr", "n_fundados", "Mp", "qt_vaga", "n_cands"]
fig = make_subplots(rows=1, cols=2, subplot_titles=["2018", "2022"])
for i, ano in enumerate(["2018", "2022"], start=1):
    corr = listas_rec[listas_rec.ano_eleicao == ano][cols_corr].corr().round(2)
    fig.add_trace(
        go.Heatmap(
            z=corr.values, x=corr.columns, y=corr.columns,
            colorscale="RdBu", zmid=0, zmin=-1, zmax=1,
            text=corr.values, texttemplate="%{text}",
            showscale=(i == 2),
        ),
        row=1, col=i,
    )
fig.update_layout(template="plotly_white", width=1000, height=500, title="Correlação entre NECr e covariadas")
fig.show()

## 11. NECr por magnitude do distrito × tipo de partido (ex-ante) × ano

`tipo_partido` é um conceito diferente de $M_p$: classifica o partido como "Competitivo" se
sua **bancada nacional** prévia (soma de `n_deputados` em todas as UFs) é ≥ 20 cadeiras —
critério de força nacional, não a magnitude partidária estadual usada acima.

In [14]:
bancada_nac = (
    bancada.groupby(["ano_eleicao", "sg_partido"])["n_deputados"]
    .sum().reset_index().rename(columns={"n_deputados": "bancada_nac"})
)
bancada_nac["ano_eleicao"] = bancada_nac["ano_eleicao"].astype(str)

listas_tp = listas_rec.merge(bancada_nac, on=["ano_eleicao", "sg_partido"], how="left")
listas_tp["bancada_nac"] = listas_tp["bancada_nac"].fillna(0)
listas_tp["tipo_partido"] = np.where(listas_tp["bancada_nac"] >= 20, "Competitivo", "Menos competitivo")
listas_tp["dm_cat"] = pd.Categorical(listas_tp["dm_cat"], categories=ordem_dm, ordered=True)

fig = px.bar(
    listas_tp.groupby(["ano_eleicao", "dm_cat", "tipo_partido"], observed=True)["NECr"].mean().reset_index(),
    x="dm_cat", y="NECr", color="tipo_partido", barmode="group", facet_col="ano_eleicao",
    category_orders={"dm_cat": ordem_dm, "tipo_partido": ["Competitivo", "Menos competitivo"]},
    labels={"dm_cat": "Magnitude do distrito", "NECr": "NECr (média)", "tipo_partido": "Tipo de partido"},
    title="NECr médio por magnitude do distrito × tipo de partido × ano",
    color_discrete_map={"Competitivo": "#1f4e79", "Menos competitivo": "#c55a11"},
)
fig.update_layout(template="plotly_white", width=1100, height=500)
fig.show()

## 12. Notas de interpretação

- `NECr = 1` → um único candidato recebe 100% dos recursos; `NECr = N` → recursos perfeitamente igualados.
- **Hipótese do $M_p$+1 (Cox 1997; Crisp et al. 2007):** sob coordenação, `NECr/(Mp+1) ≈ 1`. Valores `< 1` indicam hiperseleção; `> 1`, diluição além do esperado.
- **$M_p$ = 0 é o caso mais comum (~63% das listas) e é onde a regra falha mais** (Seção 6): partidos sem bancada estadual prévia têm `NECr/(Mp+1)` bem acima de 1 (mediana 1,87 em 2018; 3,91 em 2022) — pulverizam recursos entre vários "azarões" em vez de concentrar em um único puxador.
- Entre listas com **$M_p$ > 0**, a razão fica muito mais próxima — e em 2018 abaixo — de 1 (mediana 0,82 em 2018; 2,14 em 2022): é nesse subgrupo que a regra $M_p$+1 tem poder explicativo real, e a atenuação 2018→2022 acompanha o mesmo padrão documentado no restante da tese.
- `dm_cat` (magnitude do distrito) é mantida só como estratificador descritivo secundário.

## 13. Decomposição do NECr sob abundância de recursos: piso distribuído vs. excedente estratégico

O FEFC cresceu de ~R$1,72 bi (2018) para ~R$4,96 bi (2022), aumento nominal de ~144%. O NECr
bruto pode subir apenas por ampliação da margem extensiva (mais candidatos recebendo algum
recurso), sem que os recursos eleitoralmente decisivos tenham deixado de ser concentrados nos
candidatos prioritários. Os gráficos abaixo decompõem essa margem: participação dos Mp+1 mais
financiados, NECr do excedente acima do piso distributivo da lista, e NECr por fonte
(FEFC vs. Fundo Partidário). Ver `notebooks/3_necr.ipynb` (Seção 13) para as tabelas.

In [15]:
from cap3_necr_decomposicao import decompor_lista, CORTES_RELEVANCIA

decomp = (
    rrd
    .groupby(["ano_eleicao", "sg_uf", "sg_partido"], observed=True)
    .apply(decompor_lista, include_groups=False)
    .reset_index()
)

listas_dec = listas_rec.merge(decomp, on=["ano_eleicao", "sg_uf", "sg_partido"], how="left")
listas_dec["necr_excedente_sobre_Mpp1"] = listas_dec["necr_excedente"] / (listas_dec["Mp"] + 1)

len(listas_dec)

1434

### 13.1 Participação dos $M_p$+1 mais financiados no total da lista, por ano

In [16]:
fig = px.violin(
    listas_dec, x="ano_eleicao", y="share_top_mpp1", color="ano_eleicao",
    box=True, points=False,
    category_orders={"ano_eleicao": ["2018", "2022"]},
    labels={"share_top_mpp1": "Participação dos Mp+1 mais financiados", "ano_eleicao": "Ano"},
    title="Participação dos Mp+1 candidatos mais financiados no total de recursos da lista",
)
fig.update_layout(template="plotly_white", width=800, height=500, showlegend=False)
fig.show()

### 13.2 NECr por limiar de relevância mínima, vs. $M_p$+1

In [17]:
rows = []
for ano in ["2018", "2022"]:
    sub = listas_dec[listas_dec.ano_eleicao == ano]
    for label, corte in CORTES_RELEVANCIA.items():
        col = f"NECr_{label}"
        rows.append({
            "ano": ano, "corte": corte, "corte_label": f"≥{corte*100:g}%",
            "razao_Mpp1_mediana": (sub[col] / (sub["Mp"] + 1)).median(),
        })

tab_cortes = pd.DataFrame(rows)

fig = px.line(
    tab_cortes, x="corte_label", y="razao_Mpp1_mediana", color="ano",
    markers=True,
    category_orders={"corte_label": [f"≥{c*100:g}%" for c in CORTES_RELEVANCIA.values()]},
    labels={"corte_label": "Limiar mínimo de relevância", "razao_Mpp1_mediana": "NECr / (Mp+1)  (mediana)", "ano": "Ano"},
    title="NECr/(Mp+1) mediano conforme se eleva o limiar de relevância mínima",
)
fig.add_hline(y=1, line_dash="dot", line_color="black",
              annotation_text="previsão teórica (=1)", annotation_position="top left")
fig.update_layout(template="plotly_white", width=850, height=500)
fig.show()

### 13.3 NECr bruto vs. NECr do excedente (acima do piso distributivo da lista)

In [18]:
long_exc = listas_dec.melt(
    id_vars=["ano_eleicao"], value_vars=["NECr_sobre_Mpp1", "necr_excedente_sobre_Mpp1"],
    var_name="métrica", value_name="valor",
).dropna()
long_exc["métrica"] = long_exc["métrica"].map({
    "NECr_sobre_Mpp1": "NECr bruto / (Mp+1)",
    "necr_excedente_sobre_Mpp1": "NECr excedente / (Mp+1)",
})

fig = px.box(
    long_exc, x="métrica", y="valor", color="ano_eleicao", points=False, log_y=True,
    category_orders={"ano_eleicao": ["2018", "2022"]},
    labels={"valor": "Razão sobre (Mp+1)  [escala log]", "métrica": "", "ano_eleicao": "Ano"},
    title="NECr bruto vs. NECr do excedente estratégico, relativo a (Mp+1)",
)
fig.add_hline(y=1, line_dash="dot", line_color="black",
              annotation_text="previsão teórica (=1)", annotation_position="top left")
fig.update_layout(template="plotly_white", width=900, height=550, boxmode="group")
fig.show()

### 13.4 NECr por fonte: FEFC vs. Fundo Partidário

In [19]:
long_fonte = listas_dec.melt(
    id_vars=["ano_eleicao"], value_vars=["necr_fefc", "necr_fp"],
    var_name="fonte", value_name="valor_necr",
).dropna()
long_fonte["fonte"] = long_fonte["fonte"].map({"necr_fefc": "FEFC", "necr_fp": "Fundo Partidário"})

fig = px.box(
    long_fonte, x="fonte", y="valor_necr", color="ano_eleicao", points=False,
    category_orders={"ano_eleicao": ["2018", "2022"], "fonte": ["FEFC", "Fundo Partidário"]},
    labels={"valor_necr": "NECr", "fonte": "Fonte", "ano_eleicao": "Ano"},
    title="NECr por fonte de recurso partidário — FEFC vs. Fundo Partidário",
)
fig.update_layout(template="plotly_white", width=850, height=550, boxmode="group")
fig.show()

### 13.5 Notas de interpretação

- Se o painel 13.2 mostrar `NECr/(Mp+1)` caindo conforme o limiar sobe (aproximando-se de 1
  ou pelo menos convergindo entre 2018 e 2022), a elevação do NECr bruto reflete sobretudo uma
  cauda de candidaturas com recursos marginais.
- Se o boxplot 13.3 mostrar `NECr excedente/(Mp+1)` mais próximo de 1 (e mais estável entre
  2018 e 2022) do que `NECr bruto/(Mp+1)`, isso sustenta a leitura de que a abundância do FEFC
  ampliou principalmente o piso distribuído a candidaturas periféricas, preservando a
  concentração do excedente estratégico.
- O painel 13.4 isola se a diluição é específica do desenho do FEFC ou um padrão geral de
  recursos partidários — compare a amplitude da mudança 2018→2022 entre as duas fontes.